# Web Scraping 

In [ ]:
import requests
import re
import pandas as pd
from bs4 import BeautifulSoup


In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "accept-encoding": "gzip, deflate, br, zstd",
    "content-type": "text/plain;charset=UTF-8",
    "accept-language": "en-US,en;q=0.9"
}


In [ ]:
url = "https://www.flipkart.com/search?q=bikes%20&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off"
response = requests.get(url, headers=headers)
print(response.status_code)


In [ ]:
html_data = response.text
soup = BeautifulSoup(html_data, "html.parser")


In [ ]:
names = soup.find_all("div", class_="RG5Slk")
all_names = [x.text.strip() for x in names]


In [ ]:
specs = soup.find_all("div", class_="CMXw7N")
all_specs = [x.text.strip() for x in specs]


In [ ]:
price = soup.find_all("div", class_="hZ3P6w DeU9vF")
all_price = [x.text.strip() for x in price]


In [ ]:
rating = soup.find_all("div", class_="MKiFS6")
all_ratings = [x.text.strip() for x in rating]


In [ ]:
ratings_no = soup.find_all("span", class_="PvbNMB")
No_of_ratings = [x.text.strip() for x in ratings_no]
No_of_reviews = [x.text.strip() for x in ratings_no]


In [ ]:
min_len = min(len(all_names), len(all_specs), len(all_price), len(all_ratings), len(No_of_ratings), len(No_of_reviews))
all_names = all_names[:min_len]
all_specs = all_specs[:min_len]
all_price = all_price[:min_len]
all_ratings = all_ratings[:min_len]
No_of_ratings = No_of_ratings[:min_len]
No_of_reviews = No_of_reviews[:min_len]


In [ ]:
df = pd.DataFrame({
    "Product_Name": all_names,
    "Specifications": all_specs,
    "Price": all_price,
    "Rating": all_ratings,
    "No_of_Ratings": No_of_ratings,
    "No_of_Reviews": No_of_reviews
})


In [ ]:
df["Maximum_Power"] = df["Specifications"].str.extract(
    r"Maximum Power:\s*(.*?)(?=Engine Capacity:|Brake Front:|Range:|$)"
)
df["Engine_Capacity"] = df["Specifications"].str.extract(
    r"Engine Capacity:\s*(.*?)(?=Brake Front:|Tire Type:|Console Feature:|$)"
)
df["Brake_Front"] = df["Specifications"].str.extract(
    r"Brake Front:\s*(.*?)(?=Tire Type:|Console Feature:|$)"
)
df["Tire_Type"] = df["Specifications"].str.extract(
    r"Tire Type:\s*(.*?)(?=Console Feature:|During|$)"
)
df["Console_Feature"] = df["Specifications"].str.extract(
    r"Console Feature:\s*(.*?)(?=During|Warranty|$)"
)
df["Range"] = df["Specifications"].str.extract(
    r"Range:\s*(.*?)(?=Maximum Speed:|Charging Time:|$)"
)
df["Maximum_Speed"] = df["Specifications"].str.extract(
    r"Maximum Speed:\s*(.*?)(?=Charging Time:|Tire Type:|$)"
)
df["Charging_Time"] = df["Specifications"].str.extract(
    r"Charging Time:\s*(.*?)(?=Tire Type:|$)"
)


In [ ]:
brands = [
    "Hero", "Honda", "TVS", "Yamaha", "Bajaj",
    "Royal Enfield", "KTM", "Suzuki", "Ather",
    "VIDA", "Yezdi", "Chetak"
]

def get_brand(name):
    for brand in brands:
        if brand.lower() in name.lower():
            return brand
    return "Other"

df["Brand"] = df["Product_Name"].apply(get_brand)


In [ ]:
def bike_type(name):
    name = name.lower()
    if "electric" in name:
        return "Electric"
    elif "scooter" in name:
        return "Scooter"
    else:
        return "Motorcycle"

df["Bike_Type"] = df["Product_Name"].apply(bike_type)


In [ ]:
df_non_electric = df[df["Bike_Type"] != "Electric"]
df_non_electric.to_csv(
    "Clean_Bike_Dataset.csv",
    index=False
)
